In [ ]:
import requests
import random 
import numpy as np
import tensorflow as tf

In [239]:
book_id = 19033
urls = [
    f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
    f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
]

text = None
for url in urls:
    r = requests.get(url)
    if r.status_code == 200:
        text = r.text

        print("Downloaded: ", url)
        break

# 485 > header
# 485:54815
# 54883 total

Downloaded:  https://www.gutenberg.org/files/19033/19033-0.txt


In [240]:
# chopped header and footer and lowered everything
norm_text = text[485:54815].lower()
uniq_chars = sorted(set(norm_text))
print(f"Printing {len(uniq_chars)} characters: {uniq_chars}")

Printing 45 characters: ['\n', '\r', ' ', '!', '"', "'", '(', ')', '*', ',', '-', '.', ':', ';', '?', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'ù']


In [241]:
# global dictionaries
char2int = {}
int2char = {}

def char_to_int(chars):
    char2int.clear()          
    for i, ch in enumerate(chars):
        char2int[ch] = i
    # print(char2int)
    return char2int

def int_to_char(mapping):
    int2char.clear()
    for ch, idx in mapping.items():
        int2char[idx] = ch
    # print(int2char)
    return int2char

In [242]:
char2int = char_to_int(uniq_chars)
int2char = int_to_char(char2int)

In [243]:
def create_sequence(norm_text, char2int, seq_length, step=1):
    X = []
    y = []

    for i in range(0, len(norm_text) - seq_length, step):
        seq_in = norm_text[i : i + seq_length]
        seq_out = norm_text[i + seq_length]

        X.append([char2int[char] for char in seq_in])
        y.append(char2int[seq_out])

    return X, y

In [244]:
seq_length = 60
X, y = create_sequence(norm_text, char2int, seq_length, step=3)
X = np.array(X)
y = np.array(y)

print("training samples ", len(X))
print("First X ", X[0])
print("First y ", y[0])

training samples  18090
First X  [ 1  0  1  0 18 29 26 20 22  5 36  2 18 21 39 22 31 37 38 35 22 36  2 26
 31  2 40 32 31 21 22 35 29 18 31 21  1  0  1  0 15 26 29 29 38 36 37 35
 18 37 26 32 31 16  1  0  1  0  1  0]
First y  1


In [245]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(
        input_dim=len(uniq_chars),
        output_dim=8,
        input_length=60),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(len(uniq_chars), activation='softmax')
])

In [246]:
model.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])

In [249]:
training = model.fit(X, y, epochs=5)

Epoch 1/5
566/566 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.3228 - loss: 2.4217
Epoch 2/5
566/566 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.3506 - loss: 2.2776
Epoch 3/5
566/566 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.3717 - loss: 2.1855
Epoch 4/5
566/566 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.3920 - loss: 2.1163
Epoch 5/5
566/566 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.4106 - loss: 2.0536


In [248]:
random_idx = random.randint(0, len(X) + 20)
print(random_idx)

cur_seq = X[random_idx]
cur_seq = cur_seq.reshape(1, seq_length)

prediction = model.predict(cur_seq)

14361
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step


In [250]:
# print(prediction)

pred_char = np.argmax(prediction)
print(pred_char)

char = int2char[pred_char]
print(char)

32
o
